# Chapter 5 — Customer Churn
Queries `mart_customer_churn` from BigQuery and exports Plotly chart JSON for the webpage.

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from google.cloud import bigquery
from google.oauth2 import service_account

pio.json.config.default_engine = 'json'  # prevent binary encoding in exports

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
load_dotenv(os.path.join(project_root, '.env'))

project_id  = os.getenv('GCP_PROJECT_ID')
creds_path  = os.path.join(project_root, os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))
credentials = service_account.Credentials.from_service_account_file(creds_path)
client      = bigquery.Client(credentials=credentials, project=project_id)

OUT = os.path.join(project_root, 'outputs')
os.makedirs(OUT, exist_ok=True)
print('Connected to BigQuery ✓')

In [2]:
df = client.query(f"""
    SELECT * FROM `{project_id}.olist_raw.mart_customer_churn`
""").to_dataframe()
print(f'Total customers: {len(df):,}')
df.head()

Total customers: 95,021


,customer_unique_id,customer_state,total_orders,first_order_date,last_order_date,total_spend,avg_review_score,days_since_last_order,customer_segment,churn_status
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,2017-09-29,2017-09-29,13664.08,1.0,367,one_time,churned
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,2017-04-01,2017-04-01,7571.63,5.0,548,occasional,churned
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,2018-07-15,2018-07-15,7274.88,1.0,78,one_time,active
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,2017-02-12,2017-02-12,6929.31,5.0,596,one_time,churned
4,459bef486812aa25204be022145caa62,ES,1,2018-07-25,2018-07-25,6922.21,NaN,68,one_time,active


In [3]:
# Chart 1 — Customer Segment Breakdown (pie)
segment_counts = df['customer_segment'].value_counts().reset_index()
segment_counts.columns = ['segment', 'count']

fig1 = px.pie(
    segment_counts, names='segment', values='count',
    title='Customer Segments: One-Time vs Occasional vs Loyal',
    color='segment',
    color_discrete_map={'one_time': '#e74c3c', 'occasional': '#f39c12', 'loyal': '#2ecc71'},
    hole=0.4
)
fig1.update_layout(template='plotly_white')
fig1.show()
with open(os.path.join(OUT, 'churn_segment_breakdown.json'), 'w') as f:
    f.write(fig1.to_json())
print('Exported churn_segment_breakdown.json')

Exported churn_segment_breakdown.json


In [4]:
# Chart 2 — Churn Status Breakdown
churn_counts = df['churn_status'].value_counts().reset_index()
churn_counts.columns = ['status', 'count']

fig2 = px.bar(
    churn_counts, x='status', y='count',
    title='Customer Churn Status',
    labels={'status': 'Churn Status', 'count': 'Customers'},
    color='status',
    color_discrete_map={'active': '#2ecc71', 'at_risk': '#f39c12', 'churned': '#e74c3c'}
)
fig2.update_layout(template='plotly_white', showlegend=False)
fig2.show()
with open(os.path.join(OUT, 'churn_status_breakdown.json'), 'w') as f:
    f.write(fig2.to_json())
print('Exported churn_status_breakdown.json')

Exported churn_status_breakdown.json


In [5]:
# Chart 3 — Avg Spend by Segment
spend_by_segment = df.groupby('customer_segment').agg(
    avg_spend=('total_spend', 'mean'),
    total_customers=('customer_unique_id', 'count')
).reset_index()

fig3 = px.bar(
    spend_by_segment, x='customer_segment', y='avg_spend',
    title='Average Total Spend by Customer Segment (BRL)',
    labels={'customer_segment': 'Segment', 'avg_spend': 'Avg Spend (BRL)'},
    color='customer_segment',
    color_discrete_map={'one_time': '#e74c3c', 'occasional': '#f39c12', 'loyal': '#2ecc71'}
)
fig3.update_layout(template='plotly_white', showlegend=False)
fig3.show()
with open(os.path.join(OUT, 'churn_spend_by_segment.json'), 'w') as f:
    f.write(fig3.to_json())
print('Exported churn_spend_by_segment.json')

Exported churn_spend_by_segment.json


In [6]:
# Chart 4 — Churn Rate by State
state_churn = df.groupby('customer_state').apply(
    lambda x: pd.Series({
        'total': len(x),
        'churned': (x['churn_status'] == 'churned').sum(),
        'at_risk': (x['churn_status'] == 'at_risk').sum(),
        'active': (x['churn_status'] == 'active').sum()
    })
).reset_index()
state_churn['churn_rate'] = (state_churn['churned'] / state_churn['total'] * 100).round(1)
state_churn = state_churn.sort_values('churn_rate', ascending=False)

fig4 = px.bar(
    state_churn, x='customer_state', y='churn_rate',
    title='Customer Churn Rate by State (%)',
    labels={'customer_state': 'State', 'churn_rate': 'Churn Rate (%)'},
    color='churn_rate',
    color_continuous_scale='Reds'
)
fig4.update_layout(template='plotly_white', coloraxis_showscale=False)
fig4.show()
with open(os.path.join(OUT, 'churn_rate_by_state.json'), 'w') as f:
    f.write(fig4.to_json())
print('Exported churn_rate_by_state.json')

C:\Users\Admin\AppData\Local\Temp\ipykernel_38320\2689739630.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  state_churn = df.groupby('customer_state').apply(


Exported churn_rate_by_state.json
